# 🍎🥕 Fruit & Vegetable Image Classification

This notebook trains a **CNN image classifier** on the [Kaggle Fruit & Vegetable Dataset](https://www.kaggle.com/datasets/bcexpt1123/fruit-vegetable) using **Transfer Learning (MobileNetV2)**.

### What's Covered:
1. Dataset download & exploration
2. Data visualization (sample images, class distribution)
3. Data augmentation & preprocessing
4. Transfer learning with MobileNetV2
5. Training with callbacks
6. Essential plots: accuracy/loss curves, confusion matrix, classification report, per-class accuracy
7. Sample predictions visualization

---
**⚠️ Make sure to enable GPU:** Runtime → Change runtime type → T4 GPU

## 1. Install & Import Libraries

In [ ]:
!pip install -q kaggle

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# --- Plotting style ---
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

## 2. Download Dataset from Kaggle

Upload your `kaggle.json` API key file first. You can get it from: https://www.kaggle.com/settings → API → Create New Token

In [ ]:
from google.colab import files

# Upload kaggle.json
print("📁 Upload your kaggle.json file:")
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print("✅ Kaggle credentials configured!")

In [ ]:
# Download and extract the dataset
!kaggle datasets download -d bcexpt1123/fruit-vegetable --unzip -p /content/dataset
print("\n✅ Dataset downloaded and extracted!")

## 3. Explore Dataset Structure

In [ ]:
# --- Auto-detect dataset paths ---
# The dataset may extract into different folder structures.
# This cell auto-detects the correct train and val directories.

BASE_DIR = '/content/dataset'

def find_data_dirs(base):
    """Recursively search for 'train' and 'val' directories."""
    train_dir = None
    val_dir = None
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            full = os.path.join(root, d)
            if d.lower() == 'train' and train_dir is None:
                train_dir = full
            elif d.lower() in ('val', 'validation', 'test') and val_dir is None:
                val_dir = full
        if train_dir and val_dir:
            break
    return train_dir, val_dir

TRAIN_DIR, VAL_DIR = find_data_dirs(BASE_DIR)

if TRAIN_DIR is None or VAL_DIR is None:
    # Fallback: list what's available so user can adjust
    print("⚠️ Could not auto-detect train/val dirs. Here's the structure:")
    for root, dirs, files_list in os.walk(BASE_DIR):
        level = root.replace(BASE_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/  ({len(files_list)} files)")
        if level >= 3:
            break
else:
    print(f"📂 Train directory: {TRAIN_DIR}")
    print(f"📂 Val directory  : {VAL_DIR}")

# Get class names
class_names = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(class_names)
print(f"\n🏷️  Number of classes: {NUM_CLASSES}")
print(f"📋 Classes: {class_names}")

In [ ]:
# --- Count images per class ---
def count_images(directory):
    """Count images per class in a directory."""
    counts = {}
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))
            ])
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)

total_train = sum(train_counts.values())
total_val = sum(val_counts.values())

print(f"\n📊 Dataset Summary")
print(f"{'='*40}")
print(f"Training images  : {total_train:,}")
print(f"Validation images: {total_val:,}")
print(f"Total images     : {total_train + total_val:,}")
print(f"Number of classes: {NUM_CLASSES}")

## 4. 📊 Data Visualization

### 4.1 Class Distribution (Train vs Val)

In [ ]:
# --- Plot 1: Class Distribution Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, NUM_CLASSES * 0.35)))

# Training distribution
classes = list(train_counts.keys())
train_vals = [train_counts[c] for c in classes]
val_vals = [val_counts.get(c, 0) for c in classes]

colors_train = plt.cm.viridis(np.linspace(0.2, 0.8, len(classes)))

axes[0].barh(classes, train_vals, color=colors_train, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Number of Images')
axes[0].set_title('🏋️ Training Set Distribution', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(train_vals):
    axes[0].text(v + max(train_vals)*0.01, i, str(v), va='center', fontsize=8)

colors_val = plt.cm.plasma(np.linspace(0.2, 0.8, len(classes)))
axes[1].barh(classes, val_vals, color=colors_val, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Number of Images')
axes[1].set_title('✅ Validation Set Distribution', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
for i, v in enumerate(val_vals):
    axes[1].text(v + max(val_vals)*0.01, i, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Class distribution saved!")

### 4.2 Sample Images Per Class

In [ ]:
# --- Plot 2: Sample Images Grid ---
num_display = min(NUM_CLASSES, 20)  # Display up to 20 classes
samples_per_class = 4

fig, axes = plt.subplots(num_display, samples_per_class,
                         figsize=(samples_per_class * 3, num_display * 2.5))

display_classes = class_names[:num_display]

for i, cls in enumerate(display_classes):
    cls_path = os.path.join(TRAIN_DIR, cls)
    img_files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    selected = np.random.choice(img_files, min(samples_per_class, len(img_files)), replace=False)

    for j in range(samples_per_class):
        ax = axes[i][j] if num_display > 1 else axes[j]
        if j < len(selected):
            img = Image.open(os.path.join(cls_path, selected[j]))
            ax.imshow(img)
            if j == 0:
                ax.set_ylabel(cls, fontsize=10, fontweight='bold', rotation=0,
                              labelpad=60, ha='right')
        ax.axis('off')

plt.suptitle('🖼️ Sample Images Per Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("🖼️ Sample images saved!")

### 4.3 Image Size & Channel Distribution

In [ ]:
# --- Plot 3: Image Dimensions Analysis ---
widths, heights = [], []
sample_limit = 500  # Sample images for speed

count = 0
for cls in class_names:
    cls_path = os.path.join(TRAIN_DIR, cls)
    for img_file in os.listdir(cls_path):
        if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(os.path.join(cls_path, img_file))
                w, h = img.size
                widths.append(w)
                heights.append(h)
                count += 1
                if count >= sample_limit:
                    break
            except:
                pass
    if count >= sample_limit:
        break

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(widths, bins=30, color='#2ecc71', edgecolor='white', alpha=0.85, label='Width')
axes[0].hist(heights, bins=30, color='#e74c3c', edgecolor='white', alpha=0.65, label='Height')
axes[0].set_xlabel('Pixels')
axes[0].set_ylabel('Count')
axes[0].set_title('📐 Image Dimension Distribution', fontweight='bold')
axes[0].legend()

axes[1].scatter(widths, heights, alpha=0.4, c='#3498db', s=15, edgecolors='white', linewidth=0.3)
axes[1].set_xlabel('Width (px)')
axes[1].set_ylabel('Height (px)')
axes[1].set_title('📏 Width vs Height Scatter', fontweight='bold')
axes[1].plot([0, max(widths)], [0, max(heights)], 'r--', alpha=0.5, label='Square')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📐 Image Stats (sampled {count} images):")
print(f"   Width  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}")
print(f"   Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}")

### 4.4 Average Image Per Class (Visual Fingerprint)

In [ ]:
# --- Plot 4: Average Image per Class ---
IMG_SIZE = 128
num_avg = min(NUM_CLASSES, 15)  # Show up to 15 classes
cols = 5
rows = (num_avg + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten()

for i, cls in enumerate(class_names[:num_avg]):
    cls_path = os.path.join(TRAIN_DIR, cls)
    img_files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:30]
    imgs = []
    for f in img_files:
        try:
            img = Image.open(os.path.join(cls_path, f)).resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
            imgs.append(np.array(img))
        except:
            pass
    if imgs:
        avg_img = np.mean(imgs, axis=0).astype(np.uint8)
        axes[i].imshow(avg_img)
    axes[i].set_title(cls, fontsize=9, fontweight='bold')
    axes[i].axis('off')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('🎨 Average Image Per Class (Visual Fingerprint)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/average_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 🔧 Data Preprocessing & Augmentation

In [ ]:
# --- Hyperparameters ---
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.001

print("⚙️ Hyperparameters:")
print(f"   Image Size  : {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"   Batch Size  : {BATCH_SIZE}")
print(f"   Epochs      : {EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")

In [ ]:
# --- Data Generators with Augmentation ---
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n✅ Training batches  : {len(train_generator)}")
print(f"✅ Validation batches: {len(val_generator)}")
print(f"✅ Class indices: {train_generator.class_indices}")

### 5.1 Visualize Augmented Images

In [ ]:
# --- Plot 5: Augmented Images Preview ---
sample_batch, _ = next(train_generator)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flatten()):
    if i < len(sample_batch):
        ax.imshow(sample_batch[i])
    ax.axis('off')

plt.suptitle('🔄 Augmented Training Images Preview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 🏗️ Build Model (Transfer Learning - MobileNetV2)

In [ ]:
# --- Build Model with MobileNetV2 backbone ---
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Freeze the base model
base_model.trainable = False

# Build classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# --- Print model parameter counts ---
trainable = np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable = np.sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])

print(f"\n📈 Model Parameter Summary:")
print(f"   Trainable params    : {trainable:,}")
print(f"   Non-trainable params: {non_trainable:,}")
print(f"   Total params        : {trainable + non_trainable:,}")

## 7. 🚀 Train the Model

In [ ]:
# --- Callbacks ---
callback_list = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        '/content/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("🚀 Starting training...")

In [ ]:
# --- Train ---
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callback_list,
    verbose=1
)

print("\n✅ Training complete!")

## 8. 📈 Training History Plots

In [ ]:
# --- Plot 6: Training & Validation Accuracy/Loss Curves ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_range, history.history['accuracy'], 'o-',
             color='#2ecc71', linewidth=2, markersize=5, label='Train Accuracy')
axes[0].plot(epochs_range, history.history['val_accuracy'], 's-',
             color='#e74c3c', linewidth=2, markersize=5, label='Val Accuracy')
axes[0].set_title('📈 Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(loc='lower right', fontsize=11)
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_range, history.history['loss'], 'o-',
             color='#3498db', linewidth=2, markersize=5, label='Train Loss')
axes[1].plot(epochs_range, history.history['val_loss'], 's-',
             color='#e67e22', linewidth=2, markersize=5, label='Val Loss')
axes[1].set_title('📉 Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(loc='upper right', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
best_val_acc = max(history.history['val_accuracy'])

print(f"\n🏆 Training Results:")
print(f"   Final Train Accuracy: {final_train_acc:.4f} ({final_train_acc*100:.2f}%)")
print(f"   Final Val Accuracy  : {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
print(f"   Best Val Accuracy   : {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

## 9. 🎯 Model Evaluation

In [ ]:
# --- Evaluate on validation set ---
print("🔍 Evaluating on validation set...")
val_loss, val_accuracy = model.evaluate(val_generator, verbose=1)

print(f"\n📊 Validation Results:")
print(f"   Loss    : {val_loss:.4f}")
print(f"   Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

In [ ]:
# --- Generate Predictions ---
val_generator.reset()
y_pred_probs = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

# Get class label mapping
label_map = {v: k for k, v in val_generator.class_indices.items()}
class_labels = [label_map[i] for i in range(NUM_CLASSES)]

### 9.1 Classification Report

In [ ]:
# --- Classification Report ---
report = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True)
report_text = classification_report(y_true, y_pred, target_names=class_labels)

print("📋 Classification Report:")
print("=" * 70)
print(report_text)

### 9.2 Confusion Matrix

In [ ]:
# --- Plot 7: Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)

fig_size = max(10, NUM_CLASSES * 0.6)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels,
    ax=ax,
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 7}
)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('🎯 Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 Per-Class Accuracy Bar Chart

In [ ]:
# --- Plot 8: Per-Class Accuracy ---
per_class_acc = cm.diagonal() / cm.sum(axis=1)

# Sort by accuracy
sorted_idx = np.argsort(per_class_acc)
sorted_labels = [class_labels[i] for i in sorted_idx]
sorted_acc = per_class_acc[sorted_idx]

# Color gradient based on accuracy
colors = plt.cm.RdYlGn(sorted_acc)

fig, ax = plt.subplots(figsize=(10, max(6, NUM_CLASSES * 0.35)))
bars = ax.barh(sorted_labels, sorted_acc * 100, color=colors, edgecolor='white', linewidth=0.5)

# Add value labels
for bar, acc in zip(bars, sorted_acc):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{acc*100:.1f}%', va='center', fontsize=8, fontweight='bold')

ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('📊 Per-Class Accuracy (Sorted)', fontsize=14, fontweight='bold')
ax.set_xlim([0, 110])
ax.axvline(x=val_accuracy*100, color='red', linestyle='--', alpha=0.7,
           label=f'Overall: {val_accuracy*100:.1f}%')
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('/content/per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🏆 Best performing class : {sorted_labels[-1]} ({sorted_acc[-1]*100:.1f}%)")
print(f"🔻 Worst performing class: {sorted_labels[0]} ({sorted_acc[0]*100:.1f}%)")

### 9.4 Precision vs Recall vs F1-Score

In [ ]:
# --- Plot 9: Precision / Recall / F1-Score per class ---
metrics_data = {
    'Class': [],
    'Precision': [],
    'Recall': [],
    'F1-Score': []
}

for cls in class_labels:
    if cls in report:
        metrics_data['Class'].append(cls)
        metrics_data['Precision'].append(report[cls]['precision'])
        metrics_data['Recall'].append(report[cls]['recall'])
        metrics_data['F1-Score'].append(report[cls]['f1-score'])

df_metrics = pd.DataFrame(metrics_data)

fig, ax = plt.subplots(figsize=(max(12, NUM_CLASSES * 0.5), 6))

x = np.arange(len(df_metrics))
width = 0.25

bars1 = ax.bar(x - width, df_metrics['Precision'], width, label='Precision',
               color='#3498db', alpha=0.85)
bars2 = ax.bar(x, df_metrics['Recall'], width, label='Recall',
               color='#2ecc71', alpha=0.85)
bars3 = ax.bar(x + width, df_metrics['F1-Score'], width, label='F1-Score',
               color='#e74c3c', alpha=0.85)

ax.set_xlabel('Class')
ax.set_ylabel('Score')
ax.set_title('📊 Precision / Recall / F1-Score Per Class', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_metrics['Class'], rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.15])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/precision_recall_f1.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.5 Top Confident Predictions & Misclassifications

In [ ]:
# --- Plot 10: Sample Predictions with Confidence ---
val_generator.reset()

# Get a batch of images
batch_images, batch_labels = next(val_generator)
batch_preds = model.predict(batch_images, verbose=0)

fig, axes = plt.subplots(3, 5, figsize=(18, 10))

for i, ax in enumerate(axes.flatten()):
    if i < len(batch_images):
        ax.imshow(batch_images[i])

        true_idx = np.argmax(batch_labels[i])
        pred_idx = np.argmax(batch_preds[i])
        confidence = batch_preds[i][pred_idx] * 100

        true_label = label_map[true_idx]
        pred_label = label_map[pred_idx]

        color = '#2ecc71' if true_idx == pred_idx else '#e74c3c'
        symbol = '✅' if true_idx == pred_idx else '❌'

        ax.set_title(f"{symbol} {pred_label}\n({confidence:.1f}%)",
                     fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('🔮 Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.6 Prediction Confidence Distribution

In [ ]:
# --- Plot 11: Confidence Distribution ---
max_confidences = np.max(y_pred_probs, axis=1) * 100

correct_mask = (y_pred == y_true)
correct_conf = max_confidences[correct_mask]
incorrect_conf = max_confidences[~correct_mask]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall distribution
axes[0].hist(max_confidences, bins=50, color='#3498db', edgecolor='white',
             alpha=0.85)
axes[0].axvline(np.mean(max_confidences), color='red', linestyle='--',
                label=f'Mean: {np.mean(max_confidences):.1f}%')
axes[0].set_xlabel('Confidence (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('📊 Overall Prediction Confidence', fontweight='bold')
axes[0].legend(fontsize=11)

# Correct vs Incorrect
axes[1].hist(correct_conf, bins=30, color='#2ecc71', alpha=0.7,
             label=f'Correct (n={len(correct_conf)})', edgecolor='white')
axes[1].hist(incorrect_conf, bins=30, color='#e74c3c', alpha=0.7,
             label=f'Incorrect (n={len(incorrect_conf)})', edgecolor='white')
axes[1].set_xlabel('Confidence (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('✅❌ Confidence: Correct vs Incorrect', fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('/content/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Confidence Stats:")
print(f"   Mean confidence (all)      : {np.mean(max_confidences):.1f}%")
print(f"   Mean confidence (correct)  : {np.mean(correct_conf):.1f}%")
if len(incorrect_conf) > 0:
    print(f"   Mean confidence (incorrect): {np.mean(incorrect_conf):.1f}%")

## 10. 📊 Summary Dashboard

In [ ]:
# --- Final Summary ---
print("\n" + "=" * 60)
print("       🍎🥕 FRUIT & VEGETABLE CLASSIFIER - SUMMARY")
print("=" * 60)
print(f"\n  📂 Dataset")
print(f"     Classes          : {NUM_CLASSES}")
print(f"     Training images  : {total_train:,}")
print(f"     Validation images: {total_val:,}")
print(f"\n  🏗️ Model")
print(f"     Architecture     : MobileNetV2 (Transfer Learning)")
print(f"     Trainable params : {trainable:,}")
print(f"     Image size       : {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"\n  🎯 Performance")
print(f"     Val Accuracy     : {val_accuracy*100:.2f}%")
print(f"     Val Loss         : {val_loss:.4f}")
print(f"     Best Val Accuracy: {best_val_acc*100:.2f}%")
print(f"     Epochs trained   : {len(history.history['accuracy'])}")
print(f"\n  📊 Plots Generated")
print(f"     1. Class distribution")
print(f"     2. Sample images per class")
print(f"     3. Image dimensions analysis")
print(f"     4. Average image per class")
print(f"     5. Augmented images preview")
print(f"     6. Training accuracy/loss curves")
print(f"     7. Confusion matrix")
print(f"     8. Per-class accuracy")
print(f"     9. Precision/Recall/F1-Score")
print(f"    10. Sample predictions")
print(f"    11. Confidence distribution")
print("\n" + "=" * 60)
print("  ✅ All done! Model saved at: /content/best_model.keras")
print("=" * 60)

## 11. 💾 Save & Download Model

In [ ]:
# --- Save the final model ---
model.save('/content/fruit_veg_classifier_final.keras')
print("💾 Final model saved as: fruit_veg_classifier_final.keras")

# Download the model (uncomment to auto-download)
# from google.colab import files
# files.download('/content/fruit_veg_classifier_final.keras')

In [ ]:
# --- Predict on a single image (demo) ---
def predict_single_image(image_path):
    """Predict the class of a single image."""
    img = tf.keras.preprocessing.image.load_img(
        image_path, target_size=(IMG_HEIGHT, IMG_WIDTH)
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(predictions[0])
    confidence = predictions[0][pred_idx] * 100

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f'Predicted: {label_map[pred_idx]}\nConfidence: {confidence:.1f}%',
              fontsize=14, fontweight='bold',
              color='#2ecc71' if confidence > 80 else '#e74c3c')
    plt.axis('off')
    plt.show()

    # Top-3 predictions
    top3_idx = np.argsort(predictions[0])[-3:][::-1]
    print("\n🔝 Top-3 Predictions:")
    for idx in top3_idx:
        print(f"   {label_map[idx]:20s}: {predictions[0][idx]*100:.1f}%")

# Example usage (uncomment and provide a valid image path):
# predict_single_image('/content/dataset/labeled/val/apple/some_image.jpg')
print("\n💡 Use predict_single_image('/path/to/image.jpg') to test on any image!")